# Homework 06: Decision Trees and Ensemble Learning

**Course:** Machine Learning Zoomcamp (2026 Cohort)  
**Author:** Wisnu Anugrah Pratama  
**GitHub:** [@wapratama](https://github.com/wapratama)  
**Dataset:** 2026 Car Fuel Efficiency Dataset (`car_fuel_efficiency_2026.csv`)  
**Target:** `fuel_efficiency_mpg` (Regression Task)  

---

### Overview
This notebook contains my solutions and technical write-up for **Homework 06: Decision Trees and Ensemble Learning** as part of the [Machine Learning Zoomcamp](https://github.com/DataTalksClub/machine-learning-zoomcamp) by DataTalksClub.


## 1. Data Preparation & Split
- Fill missing values with 0,
- Perform a 60%/20%/20% Train/Validation/Test split with `random_state=1`, and 
- Vectorize using `DictVectorizer(sparse=True)`.

In [2]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

In [3]:
# Load dataset
!curl -o ../data/hw6_car_fuel_efficiency_2026.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

df = pd.read_csv('../data/hw6_car_fuel_efficiency_2026.csv')
df.head()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0   0     0   0     0     0     0  --:--:-- --:--:-- --:--:--     0
  0 635180   0  2742   0     0  3259     0   0:03:14 --:--:--  0:03:14  3268
100 635180 100 635180   0     0 595013     0   0:00:01  0:00:01 --:--:-- 596973


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
# Function to display information about missing value
def show_null(df):
    """
    Return the total missing values and the percentage of
    missing values by column.
    """
    null_count = df.isnull().sum()
    null_percentage = (null_count / df.shape[0]) * 100
    empty_count = pd.Series(((df == ' ') | (df == '')).sum())
    empty_percentage = (empty_count / df.shape[0]) * 100
    nan_count = pd.Series(((df == 'nan') | (df == 'NaN') | (df == np.nan)).sum())
    nan_percentage = (nan_count / df.shape[0]) * 100
    return pd.DataFrame({'num_missing': null_count,
                         'missing_percentage': null_percentage,
                         'num_empty': empty_count,
                         'empty_percentage': empty_percentage,
                         'nan_count': nan_count,
                         'nan_percentage': nan_percentage})

show_null(df)

,num_missing,missing_percentage,num_empty,empty_percentage,nan_count,nan_percentage
model_year,0,0.00,0,0.0,0,0.0
origin,0,0.00,0,0.0,0,0.0
fuel_type,0,0.00,0,0.0,0,0.0
drivetrain,0,0.00,0,0.0,0,0.0
num_doors,0,0.00,0,0.0,0,0.0
engine_displacement,0,0.00,0,0.0,0,0.0
num_cylinders,0,0.00,0,0.0,0,0.0
horsepower,877,8.77,0,0.0,0,0.0
vehicle_weight,0,0.00,0,0.0,0,0.0
acceleration,264,2.64,0,0.0,0,0.0


In [6]:
# Fill missing values with zeros
df = df.fillna(0)
show_null(df)

,num_missing,missing_percentage,num_empty,empty_percentage,nan_count,nan_percentage
model_year,0,0.0,0,0.0,0,0.0
origin,0,0.0,0,0.0,0,0.0
fuel_type,0,0.0,0,0.0,0,0.0
drivetrain,0,0.0,0,0.0,0,0.0
num_doors,0,0.0,0,0.0,0,0.0
engine_displacement,0,0.0,0,0.0,0,0.0
num_cylinders,0,0.0,0,0.0,0,0.0
horsepower,0,0.0,0,0.0,0,0.0
vehicle_weight,0,0.0,0,0.0,0,0.0
acceleration,0,0.0,0,0.0,0,0.0


In [7]:
# Split data: 60% Train, 20% Val, 20% Test (random_state=1)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

target = 'fuel_efficiency_mpg'
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

In [8]:
# Convert DataFrames to feature dictionaries and vectorize
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [9]:
X_train.shape

(6000, 16)

## Question 1: Decision Tree Regressor Splitting Feature
Train a decision tree regressor with `max_depth=1` and inspect the split feature.

In [11]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

# Print decision tree rules
tree_rules = export_text(dt, feature_names=list(dv.get_feature_names_out()))
print(tree_rules)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2: Validation RMSE for Random Forest (`n_estimators=10`)
Train a `RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)` and compute validation RMSE.

In [12]:
# Function for RMSE computation
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [13]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
val_rmse_q2 = rmse(y_val, y_pred_val)
print(f"Q2 Validation RMSE: {val_rmse_q2:.4f}")

Q2 Validation RMSE: 1.8332


## Question 3: Tuning `n_estimators`
Experiment with `n_estimators` in `[10, 50, 100, 150]` with `random_state=1` and select the value with the lowest RMSE.

In [14]:
n_estimators_list = [10, 50, 100, 150]

for n in n_estimators_list:
    rf_n = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    
    y_pred_n = rf_n.predict(X_val)
    val_rmse_n = rmse(y_val, y_pred_n)
    print(f"n_estimators = {n:3d} -> Validation RMSE: {val_rmse_n:.4f} (Round: {round(val_rmse_n, 3)})")

n_estimators =  10 -> Validation RMSE: 1.8332 (Round: 1.833)
n_estimators =  50 -> Validation RMSE: 1.7740 (Round: 1.774)
n_estimators = 100 -> Validation RMSE: 1.7698 (Round: 1.77)
n_estimators = 150 -> Validation RMSE: 1.7692 (Round: 1.769)


## Question 4: Tuning `max_depth`
Evaluate `max_depth` in `[10, 15, 20, 25]`. For each depth, compute mean RMSE across `n_estimators` in `[10, 50, 100, 150]`.

In [15]:
depths = [10, 15, 20, 25]
n_list = [10, 50, 100, 150]

for d in depths:
    rmses_for_d = []
    for n in n_list:
        rf_dn = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf_dn.fit(X_train, y_train)
        
        y_pred_dn = rf_dn.predict(X_val)
        rmses_for_d.append(rmse(y_val, y_pred_dn))
    
    mean_rmse_d = np.mean(rmses_for_d)
    print(f"max_depth = {d:2d} -> Mean RMSE: {mean_rmse_d:.4f}")

max_depth = 10 -> Mean RMSE: 1.7544
max_depth = 15 -> Mean RMSE: 1.7842
max_depth = 20 -> Mean RMSE: 1.7868
max_depth = 25 -> Mean RMSE: 1.7863


## Question 5: Feature Importance
Train `RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)` and extract `feature_importances_`.

In [16]:
rf_q5 = RandomForestRegressor(n_estimators=10, 
                              max_depth=20, 
                              random_state=1, 
                              n_jobs=-1)
rf_q5.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
importances = rf_q5.feature_importances_

df_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(df_imp.head(10))

                         feature  importance
9                     model_year    0.296107
15                vehicle_weight    0.203313
6             fuel_type=Gasoline    0.160712
8                     horsepower    0.078271
4            engine_displacement    0.061281
0                   acceleration    0.054998
14                    origin=USA    0.042152
2   drivetrain=Front-wheel drive    0.020062
11                     num_doors    0.019401
5               fuel_type=Diesel    0.016571


## Question 6: XGBoost Tuning (`eta=0.3` vs `eta=0.1`)
Train XGBoost Regressor for 100 rounds with `eta=0.3` and `eta=0.1`.

In [17]:
dv.get_feature_names_out()

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive',
       'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline',
       'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA',
       'vehicle_weight'], dtype=object)

In [19]:
# Clean feature names for XGBoost
features_clean = [f.replace('=', '_').replace(' ', '_').replace('-', '_') for f in dv.get_feature_names_out()]
features_clean

['acceleration',
 'drivetrain_All_wheel_drive',
 'drivetrain_Front_wheel_drive',
 'drivetrain_Rear_wheel_drive',
 'engine_displacement',
 'fuel_type_Diesel',
 'fuel_type_Gasoline',
 'fuel_type_Hybrid',
 'horsepower',
 'model_year',
 'num_cylinders',
 'num_doors',
 'origin_Asia',
 'origin_Europe',
 'origin_USA',
 'vehicle_weight']

In [20]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features_clean)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features_clean)

watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        
        'objective': 'reg:squarederror',
        'nthread': 8,
        
        'seed': 1,
        'verbosity': 0
    }
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred_xgb = model.predict(dval)
    val_rmse_xgb = rmse(y_val, y_pred_xgb)
    print(f"eta = {eta} -> Validation RMSE: {val_rmse_xgb:.4f}")


eta = 0.3 -> Validation RMSE: 1.8265
eta = 0.1 -> Validation RMSE: 1.7248
